In [ ]:
# Multi-Commodity Flow — source/sink (fractional) formulation — PuLP
# f_i(u,v) in [0,1] is the fraction of commodity i's demand routed on arc (u,v).
import pulp as pl
V     = [1, 2, 3, 4]
arcs  = [(1,2),(1,3),(2,3),(2,4),(3,4)]
cap   = {(1,2):10,(1,3):8,(2,3):4,(2,4):7,(3,4):9}
cost  = {(1,2):1,(1,3):2,(2,3):1,(2,4):3,(3,4):1}
# commodity i: (source, sink, demand)
K = {1:(1,4,10), 2:(1,4,5)}

prob = pl.LpProblem("MCF_SourceSink", pl.LpMinimize)
f = pl.LpVariable.dicts("f", [(i,u,v) for i in K for (u,v) in arcs], lowBound=0, upBound=1)
prob += pl.lpSum(cost[(u,v)]*K[i][2]*f[(i,u,v)] for i in K for (u,v) in arcs), "Total_Cost"
# (1) link capacity: sum_i f_i(u,v)*d_i <= c(u,v)
for (u,v) in arcs:
    prob += pl.lpSum(K[i][2]*f[(i,u,v)] for i in K) <= cap[(u,v)], f"Cap_{u}_{v}"
# (2)-(4) conservation: net out = +1 at source, -1 at sink, 0 elsewhere
for i,(s,t,d) in K.items():
    for u in V:
        out_ = pl.lpSum(f[(i,u,w)] for (x,w) in arcs if x==u)
        in_  = pl.lpSum(f[(i,w,u)] for (w,x) in arcs if x==u)
        rhs = 1 if u==s else (-1 if u==t else 0)
        prob += out_ - in_ == rhs, f"Bal_{i}_{u}"
prob.solve()
print("Status:", pl.LpStatus[prob.status])
for i in K:
    print(f"commodity {i} (demand {K[i][2]}):",
          {f"{u}->{v}": round(f[(i,u,v)].value(),3) for (u,v) in arcs if f[(i,u,v)].value()>1e-6})
print("Total cost =", pl.value(prob.objective))
